# Regular (flat) FSMs

A finite state machine has a finite set of states, one active at a time, and events that move it between states. A *flat* (regular) FSM has no nesting, guards or data. This notebook uses `petrilab.fsm.regular`, built on python-statemachine's `StateChart`: a traffic light and a turnstile.

```mermaid
stateDiagram-v2
    [*] --> red
    red --> green: next
    green --> yellow: next
    yellow --> red: next
```

## Traffic light
We create the machine and read its active state from `configuration` (a set of active states).

In [1]:
from petrilab.fsm.regular import TrafficLight, Turnstile

sm = TrafficLight()
seen = []
for _ in range(4):
    seen.append(next(iter(sm.configuration)).id)
    sm.send("next")
seen

['red', 'green', 'yellow', 'red']

The single event `next` cycles red, green, yellow, then back to red. The machine always has exactly one active state.

## Turnstile

```mermaid
stateDiagram-v2
    [*] --> locked
    locked --> unlocked: coin
    unlocked --> unlocked: coin
    unlocked --> locked: push
    locked --> locked: push
```

Self-transitions (`unlocked.to.itself()`) make every event valid in every state.

In [2]:
def active(sm):
    return {s.id for s in sm.configuration}


t = Turnstile()
trace = [active(t)]
for event in ["push", "coin", "coin", "push"]:
    t.send(event)
    trace.append((event, active(t)))
trace

[{'locked'},
 ('push', {'locked'}),
 ('coin', {'unlocked'}),
 ('coin', {'unlocked'}),
 ('push', {'locked'})]

Pushing a locked turnstile keeps it locked, a coin unlocks it (a second coin changes nothing), and a push locks it again.

## Invalid events
With `allow_event_without_transition = False`, an event with no matching transition raises `TransitionNotAllowed` instead of being silently ignored.

In [3]:
from statemachine.exceptions import TransitionNotAllowed

try:
    Turnstile().send("kick")
except TransitionNotAllowed as exc:
    print(type(exc).__name__, "-", exc)

TransitionNotAllowed - Can't kick when in Locked.


`kick` is not an event of the turnstile, so it is rejected loudly. This strictness makes machine bugs easy to catch.